In [91]:
from datasets import Dataset
import json
import logging
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import euclidean
from scipy.spatial.distance import cdist
import numpy as np
import evaluate

In [92]:
import os

def create_dir_if_not_exists(directory: str) -> str:
    """Создаёт директорию, если она не существует.
    
    Args:
        directory (str): Путь к директории.
    
    Returns:
        str: Абсолютный путь к созданной/существующей директории.
    """
    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Директория создана: {directory}")
    else:
        print(f"Директория уже существует: {directory}")
    return os.path.abspath(directory)

### Оценка перевода с помощью LaBSE и BLEU

Выполняется по [задаче](https://tsa04.isa.ru/ibuyanov/rudeft/-/issues/8)

In [93]:
FORMAT = "%(asctime)s %(levelname)s [%(process)s]: %(name)s: %(message)s"
logging.basicConfig(level=logging.INFO, format=FORMAT)

In [94]:
# Gold dataset
gold_dataset_file = "../rudeft_gold/gold.json"

In [95]:
# Модели - выбрать нужную, остальные закомментировать
model_dir = "deepseek-r1"
# model_dir = "deepseek-chat-v3-0324"
# model_dir = "gpt-4o-mini"
# model_dir = "qwen-2.5-72b-instruct"


In [96]:
predictions_dataset_file = f"../{model_dir}/dataset/gold.json"


In [97]:
labse_bleu_stats_dir = f"../{model_dir}/labse_bleu_stats/"

# Создаём директорию (если её нет)
abs_path = create_dir_if_not_exists(labse_bleu_stats_dir)
print(f"Абсолютный путь: {abs_path}")

Директория уже существует: ../deepseek-r1/labse_bleu_stats/
Абсолютный путь: /workspace/dpopov/deepseek-r1/labse_bleu_stats


### LaBSE

1. `count_euclid_dist_labse_between_pred_rus_and_gold_rus`

- **BLEU-like**
- считаем евклидово расстояние между ГОЛД русским переводом и Предсказанным моделью русским переводом


In [98]:
def count_euclid_dist_labse_between_pred_rus_and_gold_rus(model, gold_dataset_file, predictions_dataset_file, euclid_distances_file, stat_file):
    """Функция рассчета Евклид. расстояния между GOLD русским и предсказанным русским переводом.
    
    :param model: предзагруженная модель LaBSE.
    :param gold_dataset_file: файл GOLD датасета.
    :param predictions_dataset_file: файл датасета с переводом от LLM.
    :param euclid_distances_file: файл для записи всех значений Евклид. расстояния.
    :param stat_file: файл для записи статистики.
    
    """
    gold = Dataset.from_json(gold_dataset_file)
    pred = Dataset.from_json(predictions_dataset_file)
    
    gold_sentences = [ data['text_rus'] for data in gold ]
    pred_sentences = [ data['text_rus'] for data in pred ]
    
    assert len(gold_sentences) == len(pred_sentences), "Length of GOLD and PREDICTIONS data not equal!"
    
    gold_embeddings = model.encode(gold_sentences)
    logging.info(f"Created GOLD embeddings.")
    pred_embeddings = model.encode(pred_sentences)
    logging.info(f"Created PREDICTION embeddings.")
    
    assert len(gold_embeddings) == len(pred_embeddings), "Length of GOLD and PREDICTIONS embeddings not equal!"
    
    dist_matrix = cdist(gold_embeddings, pred_embeddings, metric='euclidean')
    # Так как у нас есть 1070 векторов в каждом наборе, диагональ матрицы dist_matrix
    # будет содержать расстояния между соответствующими векторами
    euclid_distances = np.diag(dist_matrix)
    
    stat_pred_rus_gold_rus = {
        'average': np.mean(euclid_distances),
        'min': np.min(euclid_distances),
        'max': np.max(euclid_distances)
    }
    
    stat = {
        'pred_rus_gold_rus': stat_pred_rus_gold_rus,
        'BLEU-like': stat_pred_rus_gold_rus.get("average")
    }
    
    logging.info(f"Average Euclidean distance = {stat_pred_rus_gold_rus['average']}")
    logging.info(f"    Min Euclidean distance = {stat_pred_rus_gold_rus['min']}")
    logging.info(f"    Max Euclidean distance = {stat_pred_rus_gold_rus['max']}")
        
    with open(euclid_distances_file, 'w') as f:
        json.dump(euclid_distances.tolist(), f, indent=2)

    logging.info(f"All Euclidean distances writed in file '{euclid_distances_file}'.")
    
    with open(stat_file, 'w') as f:
        json.dump(stat, f, indent=2)

    logging.info(f"Stats by Euclidean distances writed in file '{stat_file}'.")



In [99]:
model = SentenceTransformer('sentence-transformers/LaBSE')

2025-04-11 16:35:42,546 INFO [461]: sentence_transformers.SentenceTransformer: Use pytorch device_name: cuda
2025-04-11 16:35:42,547 INFO [461]: sentence_transformers.SentenceTransformer: Load pretrained SentenceTransformer: sentence-transformers/LaBSE


In [100]:
# ALL for prompt2
euclid_distances_file = f"{labse_bleu_stats_dir}euclid_distances_between_pred_rus_and_gold_rus_prompt2.json"
stat_file = f"{labse_bleu_stats_dir}euclid_distances_between_pred_rus_and_gold_rus_stat_prompt2.json"

In [101]:
count_euclid_dist_labse_between_pred_rus_and_gold_rus(model, gold_dataset_file, predictions_dataset_file, euclid_distances_file, stat_file)

Batches: 100%|██████████| 34/34 [00:00<00:00, 74.29it/s]
2025-04-11 16:35:46,647 INFO [461]: root: Created GOLD embeddings.
Batches: 100%|██████████| 34/34 [00:00<00:00, 71.31it/s]
2025-04-11 16:35:47,153 INFO [461]: root: Created PREDICTION embeddings.
2025-04-11 16:35:47,603 INFO [461]: root: Average Euclidean distance = 0.23714087571787315
2025-04-11 16:35:47,605 INFO [461]: root:     Min Euclidean distance = 0.0
2025-04-11 16:35:47,605 INFO [461]: root:     Max Euclidean distance = 0.7911501689524149
2025-04-11 16:35:47,609 INFO [461]: root: All Euclidean distances writed in file '../deepseek-r1/labse_bleu_stats/euclid_distances_between_pred_rus_and_gold_rus_prompt2.json'.
2025-04-11 16:35:47,610 INFO [461]: root: Stats by Euclidean distances writed in file '../deepseek-r1/labse_bleu_stats/euclid_distances_between_pred_rus_and_gold_rus_stat_prompt2.json'.


In [102]:
del model

2. `count_euclid_dist_labse_between_gold_eng_and_gold_rus_pred_rus`

- **Parallel comparision**

считаем евклидово расстояние между: 
- ГОЛД английским переводом и ГОЛД русским переводом
- ГОЛД английским переводом и Предсказанным моделью русским переводом

Получаем две статистики:
- ее оцениваем

In [103]:
def count_euclid_dist_labse_between_gold_eng_and_gold_rus_pred_rus(model, gold_dataset_file, predictions_dataset_file, euclid_distances_file, stat_file):
    """Функция рассчета Евклид. расстояния.
    
    Между:
    - ГОЛД английским переводом и ГОЛД русским переводом
    - ГОЛД английским переводом и Предсказанным моделью русским переводом
    
    :param model: предзагруженная модель LaBSE.
    :param gold_dataset_file: файл GOLD датасета.
    :param predictions_dataset_file: файл датасета с переводом от LLM.
    :param euclid_distances_file: файл для записи всех значений Евклид. расстояния.
    :param stat_file: файл для записи статистики.
    
    """
    gold = Dataset.from_json(gold_dataset_file)
    pred = Dataset.from_json(predictions_dataset_file)
    
    gold_rus_sentences = [ data['text_rus'] for data in gold ]
    gold_eng_sentences = [ data['text'] for data in gold ]
    pred_rus_sentences = [ data['text_rus'] for data in pred ]
    
    assert len(gold_rus_sentences) == len(pred_rus_sentences) == len(gold_eng_sentences), "Length of GOLD and PREDICTIONS data not equal!"
    
    gold_rus_embeddings = model.encode(gold_rus_sentences)
    logging.info(f"Created GOLD RUS embeddings.")
    gold_eng_embeddings = model.encode(gold_eng_sentences)
    logging.info(f"Created GOLD ENG embeddings.")
    pred_rus_embeddings = model.encode(pred_rus_sentences)
    logging.info(f"Created PREDICTION RUS embeddings.")
    
    assert len(gold_rus_embeddings) == len(pred_rus_embeddings) == len(gold_eng_embeddings), "Length of GOLD and PREDICTIONS embeddings not equal!"
    
    dist_matrix_gold_eng_gold_rus = cdist(gold_eng_embeddings, gold_rus_embeddings, metric='euclidean')
    euclid_distances_gold_eng_gold_rus = np.diag(dist_matrix_gold_eng_gold_rus)
    
    stat_gold_eng_gold_rus = {
        'average': np.mean(euclid_distances_gold_eng_gold_rus),
        'min': np.min(euclid_distances_gold_eng_gold_rus),
        'max': np.max(euclid_distances_gold_eng_gold_rus)
    }
    
    logging.info(f"Euclidean distance between GOLD ENG and GOLD RUS:")
    logging.info(f"Average = {stat_gold_eng_gold_rus['average']}")
    logging.info(f"    Min = {stat_gold_eng_gold_rus['min']}")
    logging.info(f"    Max = {stat_gold_eng_gold_rus['max']}")

    dist_matrix_gold_eng_pred_rus = cdist(gold_eng_embeddings, pred_rus_embeddings, metric='euclidean')
    euclid_distances_gold_eng_pred_rus = np.diag(dist_matrix_gold_eng_pred_rus)
    
    stat_gold_eng_pred_rus = {
        'average': np.mean(euclid_distances_gold_eng_pred_rus),
        'min': np.min(euclid_distances_gold_eng_pred_rus),
        'max': np.max(euclid_distances_gold_eng_pred_rus)
    }

    logging.info(f"Euclidean distance between GOLD ENG and PRED RUS:")
    logging.info(f"Average = {stat_gold_eng_pred_rus['average']}")
    logging.info(f"    Min = {stat_gold_eng_pred_rus['min']}")
    logging.info(f"    Max = {stat_gold_eng_pred_rus['max']}")

    
    euclid_distances = {
        'gold_eng_gold_rus': euclid_distances_gold_eng_gold_rus.tolist(),
        'gold_eng_pred_rus': euclid_distances_gold_eng_pred_rus.tolist()
    }
    
    with open(euclid_distances_file, 'w') as f:
        json.dump(euclid_distances, f, indent=2)

    logging.info(f"All Euclidean distances writed in file '{euclid_distances_file}'.")
    
    stat = {
        'gold_eng_gold_rus': stat_gold_eng_gold_rus,
        'gold_eng_pred_rus': stat_gold_eng_pred_rus,
        'Parallel comparision': {
            'AVERAGE_gold_eng_gold_rus_MINUS_gold_eng_pred_rus': stat_gold_eng_gold_rus.get('average') - stat_gold_eng_pred_rus.get('average'),
        }
    }
    
    with open(stat_file, 'w') as f:
        json.dump(stat, f, indent=2)

    logging.info(f"Stats by Euclidean distances writed in file '{stat_file}'.")



In [104]:
model = SentenceTransformer('sentence-transformers/LaBSE')

2025-04-11 16:35:47,673 INFO [461]: sentence_transformers.SentenceTransformer: Use pytorch device_name: cuda
2025-04-11 16:35:47,674 INFO [461]: sentence_transformers.SentenceTransformer: Load pretrained SentenceTransformer: sentence-transformers/LaBSE


In [105]:
# ALL for prompt2
euclid_distances_file = f"{labse_bleu_stats_dir}euclid_distances_between_gold_eng_and_gold_rus_pred_rus_prompt2.json"
stat_file = f"{labse_bleu_stats_dir}euclid_distances_between_gold_eng_and_gold_rus_pred_rus_stat_prompt2.json"

In [106]:
count_euclid_dist_labse_between_gold_eng_and_gold_rus_pred_rus(model, gold_dataset_file, predictions_dataset_file, euclid_distances_file, stat_file)

Batches: 100%|██████████| 34/34 [00:00<00:00, 68.84it/s]
2025-04-11 16:35:52,747 INFO [461]: root: Created GOLD RUS embeddings.
Batches: 100%|██████████| 34/34 [00:00<00:00, 71.06it/s]
2025-04-11 16:35:53,252 INFO [461]: root: Created GOLD ENG embeddings.
Batches: 100%|██████████| 34/34 [00:00<00:00, 68.91it/s]
2025-04-11 16:35:53,774 INFO [461]: root: Created PREDICTION RUS embeddings.
2025-04-11 16:35:54,201 INFO [461]: root: Euclidean distance between GOLD ENG and GOLD RUS:
2025-04-11 16:35:54,204 INFO [461]: root: Average = 0.48751290427504357
2025-04-11 16:35:54,205 INFO [461]: root:     Min = 0.23319765193558728
2025-04-11 16:35:54,205 INFO [461]: root:     Max = 0.8854240816176842
2025-04-11 16:35:54,622 INFO [461]: root: Euclidean distance between GOLD ENG and PRED RUS:
2025-04-11 16:35:54,623 INFO [461]: root: Average = 0.4956481256044913
2025-04-11 16:35:54,624 INFO [461]: root:     Min = 0.23319765193558728
2025-04-11 16:35:54,625 INFO [461]: root:     Max = 0.78909518211098

In [107]:
del model

### BLEU

In [108]:
def count_bleu(bleu, gold_dataset_file, predictions_dataset_file, bleu_values_file):
    """Функция рассчета BLEU.
    
    :param bleu: предзагруженная BLEU.
    :param gold_dataset_file: файл GOLD датасета.
    :param predictions_dataset_file: файл датасета с переводом от LLM.
    :param bleu_values_file: файл для записи всех значений BLEU.
    
    """

    gold = Dataset.from_json(gold_dataset_file)
    pred = Dataset.from_json(predictions_dataset_file)
    
    gold_sentences = [ data['text_rus'] for data in gold ]
    pred_sentences = [ data['text_rus'] for data in pred ]
    
    assert len(gold_sentences) == len(pred_sentences), "Length of GOLD and PREDICTIONS data not equal!"
    
    results = bleu.compute(predictions=pred_sentences, references=gold_sentences)

    logging.info(f"BLEU = {results['bleu']}")
        
    with open(bleu_values_file, 'w') as f:
        json.dump(results, f, indent=2)

    logging.info(f"BLEU results writed in file '{bleu_values_file}'.")


In [109]:
bleu = evaluate.load("bleu")

In [110]:
# ALL for prompt2
bleu_values_file = f"{labse_bleu_stats_dir}bleu_values_prompt2.json"

In [111]:
count_bleu(bleu, gold_dataset_file, predictions_dataset_file, bleu_values_file)

2025-04-11 16:35:56,788 INFO [461]: root: BLEU = 0.4625385961509151
2025-04-11 16:35:56,790 INFO [461]: root: BLEU results writed in file '../deepseek-r1/labse_bleu_stats/bleu_values_prompt2.json'.


In [112]:
del bleu